# COMP 469 — Lab 01: Vacuum World Agents — **INSTRUCTOR SOLUTION**
**CSUCI · Introduction to Artificial Intelligence · AIMA Chapters 1–2**

> ⚠️ **This is the instructor solution notebook. Do not distribute to students.**

---

## How to run this notebook in VS Code

1. Open this file in VS Code (`File → Open File`)
2. VS Code will ask *"Do you want to install the recommended extensions?"* — click **Install**
3. In the top-right corner, click **Select Kernel → Python Environments → COMP 469 (.env)**
4. Run all cells: **Run All** button at the top, or `Ctrl+Shift+P` → *"Notebook: Run All Cells"*
5. Cells run top to bottom. A green checkmark means the cell succeeded.

---

## Solution notes

All TODOs are fully implemented. Written checkpoint answers are provided as model responses.  
The notebook is designed to run clean from top to bottom after a *Restart & Run All*.


---
## 1. Setup Cell

In [ ]:
import random
from collections import deque
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import matplotlib
import matplotlib.pyplot as plt

Position = Tuple[int, int]
ACTIONS = ["Suck", "Up", "Down", "Left", "Right", "NoOp"]

random.seed(469)
print("Setup complete.")
print(f"Python: {__import__('sys').version.split()[0]}")
print(f"Matplotlib: {matplotlib.__version__}")


---
## 2. PEAS Analysis

### 2a. PEAS Table

| PEAS Component | Answer |
|---|---|
| **Performance measure** | +10 per dirty square cleaned; −1 per action taken; −2 additional per bump into a wall or obstacle; higher is better |
| **Environment** | An N×N grid of squares, each either Clean, Dirty, or Obstacle; the agent starts at (0,0) |
| **Actuators** | Suck, Up, Down, Left, Right, NoOp |
| **Sensors** | Current location (x, y) and current square status (Clean or Dirty) — nothing else |

---

### 2b. Environment Properties

| Property | Choice | Justification |
|---|---|---|
| Fully / Partially observable? | **Partially observable** | The agent perceives only its current location and square status; the rest of the grid is hidden |
| Single-agent / Multi-agent? | **Single-agent** | Only one vacuum operates; no other agents compete or cooperate |
| Deterministic / Stochastic? | **Deterministic** | The same action in the same state always produces the same next state |
| Episodic / Sequential? | **Sequential** | Each action changes the world state and affects which actions are useful later |
| Static / Dynamic? | **Static** | The environment does not change while the agent is deliberating |
| Discrete / Continuous? | **Discrete** | Finite grid, finite action set, discrete time steps |
| Known / Unknown? | **Known** | The designer knows the transition model (how actions affect the world) even though the agent cannot observe the full state |

---

### ✏️ Checkpoint 1 — Model Answer

If the sensor were upgraded to reveal the entire grid (fully observable), a reflex agent using a simple condition-action rule could be replaced by a goal-based agent that plans an optimal cleaning route from the start — eliminating wasted moves entirely. Conversely, if the performance measure penalized steps at −5 instead of −1, the serpentine reflex sweep would be far too costly and even the model-based agent's backtracking would be too slow; a planning agent that minimizes path length would become essential. In the base lab, the partial observability is what forces the distinction between reflex (react to current square), model-based (remember where you've been), and goal-based (plan toward a known goal). Changing either the sensors or the scoring changes which design is rational, even if the physical grid stays identical.


---
## 3. Vacuum World Environment — Full Implementation

### Coordinate system
```
(0,0) ── x increases right ──▶
  │
  y increases downward
  ▼
```
`"Up"` → y−1 · `"Down"` → y+1 · `"Left"` → x−1 · `"Right"` → x+1


In [ ]:
@dataclass
class Percept:
    """The only information the agent receives each step."""
    location: Position   # (x, y)
    status: str          # "Clean" or "Dirty"


class VacuumEnvironment:
    def __init__(
        self,
        width: int = 5,
        height: int = 5,
        dirt_probability: float = 0.30,
        obstacle_probability: float = 0.0,
        start: Position = (0, 0),
        seed: Optional[int] = None,
    ):
        if seed is not None:
            random.seed(seed)

        self.width = width
        self.height = height
        self.start = start
        self.agent_location = start
        self.status: Dict[Position, str] = {}
        self.obstacles: set = set()
        self.score = 0
        self.steps = 0
        self.cleaned_count = 0

        # ── TODO 1 SOLUTION: Place obstacles ──────────────────────────
        # Never place an obstacle at the start location.
        for x in range(width):
            for y in range(height):
                location = (x, y)
                if location != start and random.random() < obstacle_probability:
                    self.obstacles.add(location)

        # ── TODO 2 SOLUTION: Initialize square status ─────────────────
        for x in range(width):
            for y in range(height):
                location = (x, y)
                if location in self.obstacles:
                    self.status[location] = "Obstacle"
                else:
                    self.status[location] = (
                        "Dirty" if random.random() < dirt_probability else "Clean"
                    )
        # Agent always starts on a clean square
        self.status[start] = "Clean"

    # ── Helpers ───────────────────────────────────────────────────────
    def in_bounds(self, location: Position) -> bool:
        x, y = location
        return 0 <= x < self.width and 0 <= y < self.height

    def is_accessible(self, location: Position) -> bool:
        return self.in_bounds(location) and location not in self.obstacles

    def get_percept(self) -> Percept:
        return Percept(self.agent_location, self.status[self.agent_location])

    def get_neighbors(self, location: Position) -> Dict[str, Position]:
        """Return accessible neighbors as {action: next_location}."""
        x, y = location
        candidates = {
            "Up":    (x, y - 1),
            "Down":  (x, y + 1),
            "Left":  (x - 1, y),
            "Right": (x + 1, y),
        }
        return {
            action: next_loc
            for action, next_loc in candidates.items()
            if self.is_accessible(next_loc)
        }

    def execute_action(self, action: str) -> None:
        if action not in ACTIONS:
            raise ValueError(f"Unknown action: {action}")

        self.steps += 1
        self.score -= 1          # every action costs 1 point

        if action == "NoOp":
            return

        # ── TODO 3 SOLUTION: Suck ─────────────────────────────────────
        if action == "Suck":
            if self.status[self.agent_location] == "Dirty":
                self.status[self.agent_location] = "Clean"
                self.score += 10
                self.cleaned_count += 1
            return                # don't fall through to movement

        # ── TODO 4 SOLUTION: Movement ─────────────────────────────────
        x, y = self.agent_location
        moves = {
            "Up":    (x, y - 1),
            "Down":  (x, y + 1),
            "Left":  (x - 1, y),
            "Right": (x + 1, y),
        }
        next_location = moves[action]
        if self.is_accessible(next_location):
            self.agent_location = next_location
        else:
            self.score -= 2      # bump penalty

    # ── Utilities ─────────────────────────────────────────────────────
    def count_dirty_squares(self) -> int:
        return sum(1 for v in self.status.values() if v == "Dirty")

    def count_clean_squares(self) -> int:
        return sum(
            1 for loc, v in self.status.items()
            if v == "Clean" and loc not in self.obstacles
        )

    def is_done(self) -> bool:
        return self.count_dirty_squares() == 0

    def copy_world(self):
        return dict(self.status), set(self.obstacles)

    def load_world(self, status: Dict[Position, str], obstacles: set) -> None:
        self.status = dict(status)
        self.obstacles = set(obstacles)
        self.agent_location = self.start
        self.score = 0
        self.steps = 0
        self.cleaned_count = 0


### Smoke Test

In [ ]:
env = VacuumEnvironment(
    width=5, height=5,
    dirt_probability=0.35,
    obstacle_probability=0.10,
    seed=469,
)
print("Start location  :", env.agent_location)
print("Percept at start:", env.get_percept())
print("Dirty squares   :", env.count_dirty_squares())   # expected: 8
print("Obstacles placed:", len(env.obstacles))           # expected: 3


---
## 4. Visualization Helpers

`display_text` → ASCII art in the output cell  
`plot_environment` → matplotlib grid (agent = blue dot, dirty = dark gray, obstacle = black)


In [ ]:
def display_text(env: VacuumEnvironment, label: str = "") -> None:
    if label:
        print(f"── {label} ──")
    for y in range(env.height):
        row = []
        for x in range(env.width):
            loc = (x, y)
            if loc == env.agent_location:
                row.append("A")
            elif loc in env.obstacles:
                row.append("#")
            elif env.status[loc] == "Dirty":
                row.append("D")
            else:
                row.append(".")
        print(" ".join(row))
    print(f"Score: {env.score}  Steps: {env.steps}  Dirty left: {env.count_dirty_squares()}")


def plot_environment(env: VacuumEnvironment, title: str = "Vacuum World") -> None:
    grid = []
    for y in range(env.height):
        row = []
        for x in range(env.width):
            loc = (x, y)
            if loc in env.obstacles:
                row.append(2)
            elif env.status[loc] == "Dirty":
                row.append(1)
            else:
                row.append(0)
        grid.append(row)

    fig, ax = plt.subplots(figsize=(5, 5))
    ax.imshow(grid, cmap="Greys", vmin=0, vmax=2)
    ax.set_xticks(range(env.width))
    ax.set_yticks(range(env.height))
    ax.set_xticks([i - 0.5 for i in range(1, env.width)], minor=True)
    ax.set_yticks([i - 0.5 for i in range(1, env.height)], minor=True)
    ax.grid(which="minor", color="black", linewidth=1)
    ax.scatter(
        [env.agent_location[0]], [env.agent_location[1]],
        c="tab:blue", s=300, zorder=5, label="Agent"
    )
    ax.set_title(title)
    ax.legend(loc="upper right", fontsize=8)
    plt.tight_layout()
    plt.show()


display_text(env, label="Initial world (seed=469)")
plot_environment(env, "Initial Vacuum World (seed=469)")


---
## 5. Agent Interface and Random Baseline


In [ ]:
class Agent:
    """Abstract base — all agents implement choose_action."""
    name = "Base Agent"

    def choose_action(self, percept: Percept, environment: VacuumEnvironment) -> str:
        raise NotImplementedError


class RandomVacuumAgent(Agent):
    """
    Simple-reflex baseline. Cleans when dirty; moves randomly otherwise.
    Stateless — no instance variables.
    """
    name = "Random Vacuum Agent"

    def choose_action(self, percept: Percept, environment: VacuumEnvironment) -> str:
        if percept.status == "Dirty":
            return "Suck"
        return random.choice(["Up", "Down", "Left", "Right"])


print("RandomVacuumAgent created:", RandomVacuumAgent().name)


---
## 6. Simulation Runner


In [ ]:
def run_simulation(
    agent: Agent,
    env: VacuumEnvironment,
    max_steps: int = 100,
    verbose: bool = False,
) -> dict:
    score_history  = []
    dirty_history  = []
    action_history = []

    for step in range(max_steps):
        percept = env.get_percept()
        action  = agent.choose_action(percept, env)

        if action == "NoOp":
            break

        env.execute_action(action)

        score_history.append(env.score)
        dirty_history.append(env.count_dirty_squares())
        action_history.append(action)

        if verbose:
            print(
                f"step={step:02d}  loc={percept.location}  "
                f"status={percept.status:<6}  action={action:<6}  score={env.score}"
            )

        if env.is_done():
            break

    return {
        "agent":          agent.name,
        "score":          env.score,
        "steps":          env.steps,
        "cleaned":        env.cleaned_count,
        "dirty_left":     env.count_dirty_squares(),
        "score_history":  score_history,
        "dirty_history":  dirty_history,
        "action_history": action_history,
    }


# Quick test with verbose output so you can trace the loop
test_env = VacuumEnvironment(width=5, height=5, dirt_probability=0.35, seed=101)
test_result = run_simulation(RandomVacuumAgent(), test_env, max_steps=20, verbose=True)
print("\nFinal:", {k: v for k, v in test_result.items()
                   if k not in ("score_history","dirty_history","action_history")})


---
## 7. Simple Reflex Agent — Solution

### Design
Condition-action rule table. No `self.` state — every decision is made purely from `(x, y)` in the current percept.

**Serpentine sweep pattern:**
```
row 0 (even): → → → → ↓
row 1 (odd):  ← ← ← ← ↓
row 2 (even): → → → → ↓
...
```
The agent infers its position in the sweep from `y % 2` and `x`. No memory needed.  
When an obstacle blocks the preferred direction, a fixed priority fallback kicks in.


In [ ]:
class ReflexVacuumAgent(Agent):
    """
    TODO 6 SOLUTION — deterministic serpentine sweep.
    NO instance variables. Action depends only on current percept.
    """
    name = "Reflex Vacuum Agent"

    def choose_action(self, percept: Percept, environment: VacuumEnvironment) -> str:
        x, y = percept.location

        # Rule 1: always clean dirt immediately
        if percept.status == "Dirty":
            return "Suck"

        legal = environment.get_neighbors(percept.location)

        # Rule 2: serpentine sweep based on current (x, y)
        if y % 2 == 0:                                    # even row → sweep right
            preferred = "Right" if x < environment.width - 1 else "Down"
        else:                                             # odd row → sweep left
            preferred = "Left"  if x > 0                 else "Down"

        if preferred in legal:
            return preferred

        # Rule 3: fixed-priority fallback for obstacle-blocked rows
        for action in ["Right", "Down", "Left", "Up"]:
            if action in legal:
                return action

        return "NoOp"


# Test on an obstacle-free world to see the full sweep
reflex_env = VacuumEnvironment(width=5, height=5, dirt_probability=0.35, seed=202)
reflex_result = run_simulation(ReflexVacuumAgent(), reflex_env, max_steps=100, verbose=False)
print("Reflex agent (no obstacles):")
print({k: v for k, v in reflex_result.items()
       if k not in ("score_history","dirty_history","action_history")})

# Test with obstacles to show fallback kicking in
reflex_env2 = VacuumEnvironment(
    width=5, height=5, dirt_probability=0.35, obstacle_probability=0.15, seed=202
)
reflex_result2 = run_simulation(ReflexVacuumAgent(), reflex_env2, max_steps=100, verbose=False)
print("\nReflex agent (with obstacles):")
print({k: v for k, v in reflex_result2.items()
       if k not in ("score_history","dirty_history","action_history")})


### ✏️ Checkpoint 2 — Model Answer

The reflex agent uses only the current percept — specifically the `(x, y)` position and the dirty/clean status of the current square. It ignores everything the model-based and goal-based agents exploit: which squares have already been visited, where the remaining dirty squares are, and whether the current path is leading anywhere productive. On an obstacle-free 5×5 grid, the serpentine rule works extremely well because `(x, y)` encodes exactly where the agent is in the sweep — it visits every cell exactly once before repeating. Performance degrades on obstacle-heavy grids because obstacles interrupt the sweep and the fixed-priority fallback can send the agent into a corner or cause repeated bumps. The key limitation is that without memory, the agent cannot detect when it has cleaned everything and continues moving, burning score on wasted steps after the world is actually clean.


---
## 8. Model-Based Agent — Solution

### Design
Stores `self.visited` (set of seen positions) and `self.path_stack` (exploration history for backtracking).  
Implements depth-first coverage: always explore an unvisited neighbor first; when stuck, backtrack along the recorded path until a branch point with unvisited neighbors is found.


In [ ]:
class ModelBasedVacuumAgent(Agent):
    """
    TODOs 7 & 8 SOLUTION — visited-set + path-stack backtracking.
    Achieves full grid coverage on any connected, static grid.
    """
    name = "Model-Based Vacuum Agent"

    def __init__(self):
        self.visited:      set                  = set()
        self.known_status: Dict[Position, str]  = {}
        self.path_stack:   List[Position]       = []

    # ── Helper: convert two adjacent positions to a direction ─────────
    def action_toward(self, current: Position, target: Position) -> str:
        cx, cy = current
        tx, ty = target
        if tx == cx + 1: return "Right"
        if tx == cx - 1: return "Left"
        if ty == cy + 1: return "Down"
        if ty == cy - 1: return "Up"
        return "NoOp"

    def choose_action(self, percept: Percept, environment: VacuumEnvironment) -> str:
        location = percept.location

        # Update internal model
        self.visited.add(location)
        self.known_status[location] = percept.status

        # TODO 7 SOLUTION: clean dirty square
        if percept.status == "Dirty":
            self.known_status[location] = "Clean"
            return "Suck"

        neighbors = environment.get_neighbors(location)

        # TODO 7 SOLUTION: prefer an unvisited neighbor (DFS exploration)
        unvisited = [
            (action, next_loc)
            for action, next_loc in neighbors.items()
            if next_loc not in self.visited
        ]
        if unvisited:
            action, next_loc = unvisited[0]
            self.path_stack.append(location)   # record where we came from
            return action

        # TODO 8 SOLUTION: backtrack along path_stack when stuck
        while self.path_stack:
            prev = self.path_stack.pop()
            if prev in neighbors.values():
                return self.action_toward(location, prev)

        return "NoOp"   # fully explored — no dirty squares remain


# Test on a world with obstacles
model_env = VacuumEnvironment(
    width=5, height=5, dirt_probability=0.35, obstacle_probability=0.10, seed=303
)
model_result = run_simulation(ModelBasedVacuumAgent(), model_env, max_steps=100, verbose=False)
print("Model-based agent:")
print({k: v for k, v in model_result.items()
       if k not in ("score_history","dirty_history","action_history")})


### ✏️ Checkpoint 3 — Model Answer

The model-based agent stores two pieces of internal state: `self.visited`, a set of every position the agent has occupied, and `self.path_stack`, a stack recording the sequence of positions taken during exploration. Both are updated at every step based on the percept received, so the agent builds its model incrementally from experience rather than being given the full grid upfront. This memory fundamentally changes behavior: instead of potentially revisiting every square on every pass (as the reflex agent can), the model-based agent steers exclusively toward unvisited squares and backtracks when it reaches a dead end, achieving systematic depth-first coverage. However, the agent does not have a complete model — it only knows the status of squares it has personally visited, and it does not know the grid dimensions or obstacle locations in advance; it discovers those implicitly through `get_neighbors()`. The remaining weakness is that the backtracking scheme assumes the grid is connected; if obstacles create an isolated region the agent never enters, those squares will never be cleaned and the agent will terminate via `NoOp` without knowing dirty squares still exist.


---
## 9. Goal-Based Agent — Solution

### Design
Re-plans at every step using BFS. Always navigates to the nearest reachable dirty square.  
Acknowledged simplification: the agent inspects `environment.status` directly (full observability).


In [ ]:
class GoalBasedVacuumAgent(Agent):
    """
    TODO 9 SOLUTION — BFS to nearest dirty square, re-planned every step.
    """
    name = "Goal-Based Vacuum Agent"

    def choose_action(self, percept: Percept, environment: VacuumEnvironment) -> str:
        if percept.status == "Dirty":
            return "Suck"

        dirty_locations = [
            loc for loc, status in environment.status.items()
            if status == "Dirty"
        ]

        if not dirty_locations:
            return "NoOp"

        path = self.bfs_to_nearest_dirty(percept.location, dirty_locations, environment)
        return path[0] if path else "NoOp"

    def bfs_to_nearest_dirty(
        self,
        start:           Position,
        dirty_locations: List[Position],
        environment:     VacuumEnvironment,
    ) -> List[str]:
        """
        BFS from `start` to the nearest position in `dirty_locations`.
        Returns the action sequence, or [] if none is reachable.
        """
        dirty_set = set(dirty_locations)
        frontier  = deque([(start, [])])   # (location, path_so_far)
        visited   = {start}                # ← must be initialised BEFORE the loop

        while frontier:
            location, path_so_far = frontier.popleft()

            if location in dirty_set:
                return path_so_far         # shortest path found

            for action, next_loc in environment.get_neighbors(location).items():
                if next_loc not in visited:
                    visited.add(next_loc)
                    frontier.append((next_loc, path_so_far + [action]))

        return []   # no reachable dirty square


# Test on same obstacle world as model-based
goal_env = VacuumEnvironment(
    width=5, height=5, dirt_probability=0.35, obstacle_probability=0.10, seed=404
)
goal_result = run_simulation(GoalBasedVacuumAgent(), goal_env, max_steps=100, verbose=False)
print("Goal-based agent:")
print({k: v for k, v in goal_result.items()
       if k not in ("score_history","dirty_history","action_history")})


### ✏️ Checkpoint 4 — Model Answer

The goal of this agent is a desired world state in which every reachable square is clean; at each step, the sub-goal is to reach and clean the nearest dirty square as efficiently as possible. BFS is appropriate because the grid is an unweighted graph — every move costs the same 1 point — so BFS guarantees the minimum-step (and therefore minimum-movement-cost) path to the nearest dirty square, which aligns directly with the performance measure. The goal-based agent uses information the model-based agent never has: the complete current `environment.status`, giving it the exact location of every remaining dirty square before it takes a single step. This means the comparison is fundamentally asymmetric — the goal-based agent is solving a simpler, fully-observable problem, while the model-based agent must discover dirt through exploration under partial observability. A truly fair comparison would require the goal-based agent to plan only over squares it has already perceived, which is what a realistic goal-based agent (combined with a world model) would do in a partially observable setting.


---
## 10. Controlled Experiments


In [ ]:
def evaluate_agents(
    agent_factories,
    trials:              int   = 30,
    max_steps:           int   = 100,
    width:               int   = 5,
    height:              int   = 5,
    dirt_probability:    float = 0.35,
    obstacle_probability: float = 0.10,
) -> List[dict]:
    """
    Run each agent on identical worlds across `trials` trials.
    copy_world/load_world ensures every agent sees exactly the same grid.
    """
    rows = []
    for trial in range(trials):
        base_env = VacuumEnvironment(
            width=width, height=height,
            dirt_probability=dirt_probability,
            obstacle_probability=obstacle_probability,
            seed=1000 + trial,
        )
        status, obstacles = base_env.copy_world()

        for make_agent in agent_factories:
            env = VacuumEnvironment(width=width, height=height)
            env.load_world(status, obstacles)
            agent = make_agent()
            result = run_simulation(agent, env, max_steps=max_steps)
            result["trial"] = trial
            rows.append(result)

    return rows


def summarize_results(results: List[dict]) -> Dict[str, dict]:
    summary = {}
    for name in sorted({r["agent"] for r in results}):
        rows = [r for r in results if r["agent"] == name]
        summary[name] = {
            "avg_score":      sum(r["score"]      for r in rows) / len(rows),
            "avg_steps":      sum(r["steps"]      for r in rows) / len(rows),
            "avg_cleaned":    sum(r["cleaned"]    for r in rows) / len(rows),
            "avg_dirty_left": sum(r["dirty_left"] for r in rows) / len(rows),
        }
    return summary


agent_factories = [
    lambda: RandomVacuumAgent(),
    lambda: ReflexVacuumAgent(),
    lambda: ModelBasedVacuumAgent(),
    lambda: GoalBasedVacuumAgent(),
]

results = evaluate_agents(agent_factories, trials=30, max_steps=100)
summary = summarize_results(results)

# Pretty-print summary table
header = f"{'Agent':<30} {'Avg Score':>10} {'Avg Steps':>10} {'Avg Cleaned':>12} {'Avg Dirty Left':>15}"
print(header)
print("─" * len(header))
for name, s in summary.items():
    print(
        f"{name:<30} {s['avg_score']:>10.1f} {s['avg_steps']:>10.1f} "
        f"{s['avg_cleaned']:>12.1f} {s['avg_dirty_left']:>15.1f}"
    )


---
## 11. Results Plots

In [ ]:
def plot_bar(
    summary: Dict[str, dict],
    metric: str,
    title: str,
    ylabel: str,
    higher_is_better: bool = True,
    color_best: str = "darkorange",
    color_rest: str = "steelblue",
) -> None:
    names  = list(summary.keys())
    values = [summary[n][metric] for n in names]
    best   = values.index(max(values) if higher_is_better else min(values))
    colors = [color_best if i == best else color_rest for i in range(len(names))]

    fig, ax = plt.subplots(figsize=(10, 4))
    bars = ax.bar(names, values, color=colors)
    for bar, val in zip(bars, values):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + abs(max(values) - min(values)) * 0.02,
            f"{val:.1f}", ha="center", va="bottom", fontsize=9, fontweight="bold"
        )
    ax.set_title(title, fontsize=12)
    ax.set_ylabel(ylabel)
    ax.tick_params(axis="x", rotation=15)
    ax.set_ylim(
        min(0, min(values)) - abs(max(values) - min(values)) * 0.15,
        max(values) + abs(max(values) - min(values)) * 0.15,
    )
    note = "(higher = better)" if higher_is_better else "(lower = better)"
    ax.set_xlabel(f"Agent type  {note}", fontsize=9, color="gray")
    plt.tight_layout()
    plt.show()


plot_bar(
    summary, "avg_score",
    "Average Score by Agent  (30 trials · 5×5 grid · 35% dirt · 10% obstacles)",
    "Average final score",
    higher_is_better=True,
)

plot_bar(
    summary, "avg_dirty_left",
    "Average Dirty Squares Remaining by Agent",
    "Avg dirty squares left at end",
    higher_is_better=False,
)

plot_bar(
    summary, "avg_steps",
    "Average Steps Used by Agent",
    "Avg steps",
    higher_is_better=False,
)

plot_bar(
    summary, "avg_cleaned",
    "Average Squares Successfully Cleaned by Agent",
    "Avg squares cleaned",
    higher_is_better=True,
)


### Checkpoint 5 — Model Answer

*(Exact numbers will vary slightly by Python version; these reflect the reference run.)*

The goal-based agent achieves the highest average score and leaves the fewest dirty squares — it cleans nearly all reachable dirt every trial because BFS always finds the shortest path to the next dirty square, eliminating wasted movement. Interestingly, it does not necessarily use the fewest steps; the model-based agent with backtracking sometimes terminates earlier on sparse-dirt worlds because it declares `NoOp` once its stack is exhausted, while the goal-based agent keeps planning until truly done. Obstacles hurt the reflex agent disproportionately because its serpentine sweep breaks at an obstacle and the fixed-priority fallback can trap it in a sub-region it has already cleaned, burning steps on moves that score nothing. The results are not a clean comparison between agent architectures, however, because the goal-based agent has full observability — an advantage the others do not share. If bumps cost −10 instead of −2, the reflex agent's ranking would fall the most, since it has no model to predict which moves are safe; the goal-based agent would be least affected because BFS through `get_neighbors()` already routes around obstacles entirely.


---
## 12. Extension — Option B: Dynamic Environment

Demonstrates how each agent handles a world where clean squares can become dirty again.


In [ ]:
class DynamicVacuumEnvironment(VacuumEnvironment):
    """
    Extension Option B: after each action, each clean square has a small
    probability of becoming dirty again (simulates ongoing mess generation).
    """
    def __init__(self, *args, redirt_probability: float = 0.02, **kwargs):
        super().__init__(*args, **kwargs)
        self.redirt_probability = redirt_probability

    def execute_action(self, action: str) -> None:
        super().execute_action(action)
        # After every action, randomly re-dirty some clean squares
        for loc, status in self.status.items():
            if status == "Clean" and loc not in self.obstacles:
                if random.random() < self.redirt_probability:
                    self.status[loc] = "Dirty"


# Compare model-based vs goal-based on dynamic world
print("Dynamic environment (2% re-dirt per step):")
print()
for AgentClass in [RandomVacuumAgent, ReflexVacuumAgent, ModelBasedVacuumAgent, GoalBasedVacuumAgent]:
    dyn_env = DynamicVacuumEnvironment(
        width=5, height=5, dirt_probability=0.35,
        obstacle_probability=0.10, redirt_probability=0.02, seed=777,
    )
    result = run_simulation(AgentClass(), dyn_env, max_steps=100)
    print(f"  {result['agent']:<30}  score={result['score']:>5}  "
          f"cleaned={result['cleaned']:>3}  dirty_left={result['dirty_left']:>2}")


---
## 13. Final Reflection — Model Answer

In this Vacuum World, an agent is rational (AIMA §2.2) if it selects actions that maximize its **expected** performance measure given its percept sequence and any built-in prior knowledge — not if it happens to achieve the best outcome in one trial. By that definition, the highest-scoring agent in the experiments is not automatically the most rational. The goal-based agent consistently scores highest, but it exploits full observability of `environment.status`, a capability the other agents do not have; a goal-based agent operating under the same partial observability as the model-based agent would need to integrate exploration with planning and would not automatically dominate. The model-based agent with backtracking is arguably the most rational among the agents that operate under realistic partial observability: given only the current percept and its accumulated visit history, systematic DFS coverage is close to optimal — it wastes no move on a square it knows it has already cleaned, and the backtracking guarantees it will not get permanently stuck in an explored sub-region. The reflex agent can be rational on small, obstacle-free grids where the serpentine sweep visits every cell before the step budget runs out, but it becomes irrational on grids with obstacles because it has no way to detect or recover from the situations where the sweep breaks down. This illustrates the core AIMA insight: rationality is always relative to the performance measure, the percept available, and the prior knowledge built into the agent design — there is no universally rational agent, only agents that are rational with respect to a specific task environment.

---

## Submission Checklist

- [x] Restarts and runs top to bottom without errors
- [x] PEAS table complete
- [x] Environment property table complete with justifications
- [x] Smoke test shows nonzero dirty and obstacle counts
- [x] Reflex agent uses deterministic rule, no stored state
- [x] Model-based agent stores and uses `self.visited`
- [x] Goal-based BFS returns valid paths
- [x] Experiment summary visible
- [x] All four plots visible and labeled
- [x] All five checkpoints answered
- [x] Final reflection complete
